In [1]:
!nvidia-smi

Tue Sep 22 08:45:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U "torchao>=0.16.0"

In [3]:
# !pip install -q -U "torchao>=0.16.0"

In [4]:
!pip install -q -U transformers peft datasets accelerate

In [5]:
import torch
import transformers
import peft
import datasets
import accelerate

print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("PEFT         :", peft.__version__)
print("Datasets     :", datasets.__version__)
print("Accelerate   :", accelerate.__version__)

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch      : 2.11.0+cu128
Transformers : 5.17.0
PEFT         : 0.21.0
Datasets     : 5.0.1
Accelerate   : 1.15.0

GPU available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [6]:
import os
import gc
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

In [7]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Model:", MODEL_ID)
print("Device:", DEVICE)

Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Vocabulary size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Vocabulary size: 151643
PAD token: <|endoftext|>
EOS token: <|im_end|>


In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model loaded successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully.


In [10]:
messages = [
    {
        "role": "user",
        "content": "Explain what a SQL JOIN does in one simple sentence."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response)

A SQL JOIN (also known as an INNER JOIN) is used to combine rows from two or more tables based on matching values in their respective columns. This operation allows for the selection of specific records from each table while retaining only those that match across all relevant fields, effectively filtering out duplicate entries and providing a clear view of relationships between different sets of data.


              SAME TEST SET
                    │
          ┌─────────┴─────────┐
          ↓                   ↓
   Base Qwen 0.5B        Fine-tuned LoRA
          │                   │
      Baseline           Task-specific
          │                   │
          └─────────┬─────────┘
                    ↓
               Improvement?
                    ↓
             Add 3 adapters
                    ↓
                 Router
                    ↓
             Dynamic selection

In [11]:
!pip install -q rouge-score

In [12]:
from datasets import load_dataset

SEED = 42

# -------------------------
# SQL
# -------------------------
sql_all = load_dataset(
    "b-mc2/sql-create-context",
    split="train"
)

# SQL dataset has only train, so create our own fixed split.
sql_all = sql_all.shuffle(seed=SEED)

sql_train = sql_all.select(range(0, 1000))
sql_val   = sql_all.select(range(1000, 1100))
sql_test  = sql_all.select(range(1100, 1200))


# -------------------------
# Summarization
# -------------------------
samsum = load_dataset("knkarthick/samsum")

sam_train = samsum["train"].select(range(1000))
sam_val   = samsum["validation"].select(range(100))
sam_test  = samsum["test"].select(range(100))


# -------------------------
# Math
# -------------------------
gsm8k = load_dataset("openai/gsm8k", "main")

math_train = gsm8k["train"].select(range(1000))
math_val   = gsm8k["train"].select(range(1000, 1100))
math_test  = gsm8k["test"].select(range(100))


print("SQL:", len(sql_train), len(sql_val), len(sql_test))
print("SAMSum:", len(sam_train), len(sam_val), len(sam_test))
print("GSM8K:", len(math_train), len(math_val), len(math_test))

SQL: 1000 100 100
SAMSum: 1000 100 100
GSM8K: 1000 100 100


In [13]:
def sql_prompt(row):
    return f"""Generate the SQL query that answers the question.

Database schema:
{row["context"]}

Question:
{row["question"]}

Return only the SQL query."""

In [14]:
def summary_prompt(row):
    return f"""Summarize the following conversation in 1-3 concise sentences.

Conversation:
{row["dialogue"]}

Return only the summary."""

In [15]:
def math_prompt(row):
    return f"""Solve the following math word problem.

Problem:
{row["question"]}

Give the final numerical answer clearly."""

In [16]:
tokenizer.padding_side = "left"

def generate_batch(prompts, max_new_tokens=128, batch_size=8):
    all_outputs = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]

        conversations = [
            [{"role": "user", "content": prompt}]
            for prompt in batch_prompts
        ]

        texts = [
            tokenizer.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=True
            )
            for conversation in conversations
        ]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )

        inputs = {
            key: value.to(model.device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        input_length = inputs["input_ids"].shape[1]

        new_tokens = generated[:, input_length:]

        decoded = tokenizer.batch_decode(
            new_tokens,
            skip_special_tokens=True
        )

        all_outputs.extend(decoded)

    return all_outputs

In [17]:
sql_examples = [sql_prompt(sql_test[i]) for i in range(3)]
sum_examples = [summary_prompt(sam_test[i]) for i in range(3)]
math_examples = [math_prompt(math_test[i]) for i in range(3)]

sql_outputs = generate_batch(
    sql_examples,
    max_new_tokens=128,
    batch_size=3
)

sum_outputs = generate_batch(
    sum_examples,
    max_new_tokens=128,
    batch_size=3
)

math_outputs = generate_batch(
    math_examples,
    max_new_tokens=256,
    batch_size=3
)

print("===== SQL =====")
for i, out in enumerate(sql_outputs):
    print(f"\nExample {i+1}:\n{out}")

print("\n===== SUMMARY =====")
for i, out in enumerate(sum_outputs):
    print(f"\nExample {i+1}:\n{out}")

print("\n===== MATH =====")
for i, out in enumerate(math_outputs):
    print(f"\nExample {i+1}:\n{out}")

===== SQL =====

Example 1:
```sql
SELECT vs_terran FROM table_name_12 WHERE vs_zerg = 69;
```

Example 2:
```sql
SELECT venue FROM table_name_99 WHERE away_team = 'Collingwood';
```

Example 3:
```sql
SELECT DISTINCT construction FROM table_22180353_1 WHERE registration = 'HB-HOS';
```

===== SUMMARY =====

Example 1:
Hannah and Amanda discuss finding Betty's phone number. Amanda calls Larry to ask if she can find Betty's number. Hannah declines to call Larry as she doesn't know him well. They agree that Hannah should text Betty instead.

Example 2:
The conversation is about Eric and Rob discussing their favorite TV show "MACHINE." They both enjoy watching the show, which features American humor and stands-up comedy. Rob mentions that there are some of Eric's stand-up videos on YouTube, and they decide to watch them together.

Example 3:
Bob suggests Lenny choose between two pairs of pants based on their quality and outfit options.

===== MATH =====

Example 1:
To determine how much J

In [18]:
sql_test_prompts = [
    sql_prompt(row)
    for row in sql_test
]

sum_test_prompts = [
    summary_prompt(row)
    for row in sam_test
]

math_test_prompts = [
    math_prompt(row)
    for row in math_test
]

In [19]:
sql_baseline = generate_batch(
    sql_test_prompts,
    max_new_tokens=128,
    batch_size=8
)

sum_baseline = generate_batch(
    sum_test_prompts,
    max_new_tokens=128,
    batch_size=8
)

math_baseline = generate_batch(
    math_test_prompts,
    max_new_tokens=256,
    batch_size=8
)

In [20]:
import re

def normalize_sql(text):
    text = text.strip()

    text = re.sub(
        r"```(?:sql)?",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = text.replace("```", "")

    text = re.sub(r"\s+", " ", text)

    return text.strip().lower()

In [21]:
sql_correct = 0

for row, prediction in zip(sql_test, sql_baseline):
    expected = normalize_sql(row["answer"])
    predicted = normalize_sql(prediction)

    if predicted == expected:
        sql_correct += 1

sql_accuracy = sql_correct / len(sql_test)

print(f"SQL exact-match accuracy: {sql_accuracy:.3f}")

SQL exact-match accuracy: 0.000


In [22]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rougeL_scores = []

for row, prediction in zip(sam_test, sum_baseline):

    reference = row["summary"]

    scores = scorer.score(
        reference,
        prediction
    )

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
avg_rougeL = sum(rougeL_scores) / len(rougeL_scores)

print(f"ROUGE-1 F1: {avg_rouge1:.3f}")
print(f"ROUGE-L F1: {avg_rougeL:.3f}")

ROUGE-1 F1: 0.344
ROUGE-L F1: 0.263


In [23]:
def extract_gsm8k_answer(text):
    text = str(text).strip()

    if "####" in text:
        text = text.split("####")[-1]

    # Remove commas and common formatting.
    text = text.replace(",", "")
    text = text.strip()

    match = re.findall(
        r"[-+]?\d+(?:\.\d+)?(?:/\d+)?",
        text
    )

    if not match:
        return None

    return match[-1]

In [24]:
math_correct = 0

for row, prediction in zip(math_test, math_baseline):

    expected = extract_gsm8k_answer(row["answer"])
    predicted = extract_gsm8k_answer(prediction)

    if expected is not None and predicted == expected:
        math_correct += 1

math_accuracy = math_correct / len(math_test)

print(f"GSM8K final-answer accuracy: {math_accuracy:.3f}")

GSM8K final-answer accuracy: 0.220


In [25]:
math_correct = 0

for row, prediction in zip(math_test, math_baseline):

    expected = extract_gsm8k_answer(row["answer"])
    predicted = extract_gsm8k_answer(prediction)

    if expected is not None and predicted == expected:
        math_correct += 1

math_accuracy = math_correct / len(math_test)

print(f"GSM8K final-answer accuracy: {math_accuracy:.3f}")

GSM8K final-answer accuracy: 0.220


In [26]:
import gc
import torch

del model

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

GPU cache cleared.


In [27]:
from peft import LoraConfig

LORA_CONFIG = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print(LORA_CONFIG)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'k_proj', 'o_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


Create task-specific training examples

We want the model to learn:

In [28]:
def build_sql_example(row):
    prompt = f"""Generate the SQL query that answers the question.

Database schema:
{row["context"]}

Question:
{row["question"]}

Return only the SQL query."""

    target = row["answer"]

    return prompt, target

In [29]:
def build_summary_example(row):
    prompt = f"""Summarize the following conversation in 1-3 concise sentences.

Conversation:
{row["dialogue"]}

Return only the summary."""

    target = row["summary"]

    return prompt, target

In [30]:
def build_math_example(row):
    prompt = f"""Solve the following math word problem.

Problem:
{row["question"]}

Give the final answer with your reasoning."""

    target = row["answer"]

    return prompt, target

Convert examples into ChatML training format

USER:
<task>

ASSISTANT:
<correct answer>

In [31]:
def tokenize_instruction_example(prompt, target, max_length=1024):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    full_messages = [
        {
            "role": "user",
            "content": prompt
        },
        {
            "role": "assistant",
            "content": target
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]

    # Keep the complete sequence within max_length.
    full_tokens = full_tokens[:max_length]

    prompt_length = min(
        len(prompt_tokens),
        len(full_tokens)
    )

    labels = (
        [-100] * prompt_length
        + full_tokens[prompt_length:]
    )

    attention_mask = [1] * len(full_tokens)

    return {
        "input_ids": full_tokens,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [32]:
def prepare_dataset(dataset, example_builder):

    processed = []

    for row in dataset:
        prompt, target = example_builder(row)

        processed.append(
            tokenize_instruction_example(
                prompt,
                target
            )
        )

    from datasets import Dataset

    return Dataset.from_list(processed)

In [33]:
sql_train_tokenized = prepare_dataset(
    sql_train,
    build_sql_example
)

sql_val_tokenized = prepare_dataset(
    sql_val,
    build_sql_example
)


summary_train_tokenized = prepare_dataset(
    sam_train,
    build_summary_example
)

summary_val_tokenized = prepare_dataset(
    sam_val,
    build_summary_example
)


math_train_tokenized = prepare_dataset(
    math_train,
    build_math_example
)

math_val_tokenized = prepare_dataset(
    math_val,
    build_math_example
)

In [34]:
print(sql_train_tokenized)
print(summary_train_tokenized)
print(math_train_tokenized)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})


Create the padding collator

Sequences have different lengths, so we pad them dynamically inside each batch.

In [35]:
class CausalLMDataCollator:

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):

        max_length = max(
            len(x["input_ids"])
            for x in features
        )

        input_ids = []
        attention_mask = []
        labels = []

        for feature in features:

            length = len(feature["input_ids"])
            padding = max_length - length

            input_ids.append(
                feature["input_ids"]
                + [self.pad_token_id] * padding
            )

            attention_mask.append(
                feature["attention_mask"]
                + [0] * padding
            )

            labels.append(
                feature["labels"]
                + [-100] * padding
            )

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_mask,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }


data_collator = CausalLMDataCollator(tokenizer)

In [36]:
from transformers import AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

def load_lora_model():

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    base_model.config.use_cache = False

    lora_model = get_peft_model(
        base_model,
        LORA_CONFIG
    )

    lora_model.print_trainable_parameters()

    return lora_model

Training configuration

We will use:

1,000 examples/task
2 epochs
batch size 4
gradient accumulation 4
FP16
learning rate 2e-4

In [37]:
from transformers import TrainingArguments

TRAIN_ARGS = TrainingArguments(
    output_dir="/content/tmp_lora_training",

    num_train_epochs=2,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=25,

    eval_strategy="epoch",

    save_strategy="no",

    report_to="none",

    remove_unused_columns=False
)

Train SQL LoRA

In [38]:
from transformers import Trainer

sql_model = load_lora_model()

sql_trainer = Trainer(
    model=sql_model,
    args=TRAIN_ARGS,
    train_dataset=sql_train_tokenized,
    eval_dataset=sql_val_tokenized,
    data_collator=data_collator
)

sql_trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


Epoch,Training Loss,Validation Loss
1,0.100140,0.104236
2,0.063332,0.100135


TrainOutput(global_step=126, training_loss=0.09175843044760681, metrics={'train_runtime': 99.99, 'train_samples_per_second': 20.002, 'train_steps_per_second': 1.26, 'total_flos': 562196290123776.0, 'train_loss': 0.09175843044760681, 'epoch': 2.0})

save model

In [39]:
SQL_ADAPTER_PATH = "/content/adapters/sql"

sql_model.save_pretrained(
    SQL_ADAPTER_PATH
)

tokenizer.save_pretrained(
    SQL_ADAPTER_PATH
)

print("SQL adapter saved:", SQL_ADAPTER_PATH)

SQL adapter saved: /content/adapters/sql


Free SQL model

In [40]:
del sql_trainer
del sql_model

gc.collect()
torch.cuda.empty_cache()

print("SQL model removed from GPU.")

SQL model removed from GPU.


Train Summarization LoRA

In [41]:
summary_model = load_lora_model()

summary_trainer = Trainer(
    model=summary_model,
    args=TRAIN_ARGS,
    train_dataset=summary_train_tokenized,
    eval_dataset=summary_val_tokenized,
    data_collator=data_collator
)

summary_trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


Epoch,Training Loss,Validation Loss
1,1.533029,1.411879
2,1.396160,1.398293


TrainOutput(global_step=126, training_loss=1.4615930280988179, metrics={'train_runtime': 156.3088, 'train_samples_per_second': 12.795, 'train_steps_per_second': 0.806, 'total_flos': 1440254295828480.0, 'train_loss': 1.4615930280988179, 'epoch': 2.0})

In [42]:
SUMMARY_ADAPTER_PATH = "/content/adapters/summary"

summary_model.save_pretrained(
    SUMMARY_ADAPTER_PATH
)

tokenizer.save_pretrained(
    SUMMARY_ADAPTER_PATH
)

print("Summary adapter saved:", SUMMARY_ADAPTER_PATH)

Summary adapter saved: /content/adapters/summary


In [43]:
del summary_trainer
del summary_model

gc.collect()
torch.cuda.empty_cache()

In [44]:
math_model = load_lora_model()

math_trainer = Trainer(
    model=math_model,
    args=TRAIN_ARGS,
    train_dataset=math_train_tokenized,
    eval_dataset=math_val_tokenized,
    data_collator=data_collator
)

math_trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


Epoch,Training Loss,Validation Loss
1,0.443860,0.431724
2,0.423438,0.428058


TrainOutput(global_step=126, training_loss=0.4387302947422815, metrics={'train_runtime': 139.503, 'train_samples_per_second': 14.337, 'train_steps_per_second': 0.903, 'total_flos': 1305636999828480.0, 'train_loss': 0.4387302947422815, 'epoch': 2.0})

In [45]:
MATH_ADAPTER_PATH = "/content/adapters/math"

math_model.save_pretrained(
    MATH_ADAPTER_PATH
)

tokenizer.save_pretrained(
    MATH_ADAPTER_PATH
)

print("Math adapter saved:", MATH_ADAPTER_PATH)

Math adapter saved: /content/adapters/math


In [46]:
del math_trainer
del math_model

gc.collect()
torch.cuda.empty_cache()

print("All adapter training completed.")

All adapter training completed.


In [47]:
import os

for path in [
    SQL_ADAPTER_PATH,
    SUMMARY_ADAPTER_PATH,
    MATH_ADAPTER_PATH
]:
    print(
        path,
        "→",
        os.listdir(path)
    )

/content/adapters/sql → ['tokenizer.json', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja']
/content/adapters/summary → ['tokenizer.json', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja']
/content/adapters/math → ['tokenizer.json', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja']


### 3.1 — Robust multi-adapter evaluation (fixed)

**What was breaking Phase 3, and what changed:**

1. **Train/test prompt mismatch — the main reason Math-LoRA barely beat the base model.** Training used the prompt built by `build_math_example()`, but evaluation used the differently-worded `math_prompt()` ("Give the final numerical answer clearly." vs. "Give the final answer with your reasoning."). That wording shift meant the math adapter was tested on a prompt shape it never actually trained on. Fixed by rebuilding all three eval prompt sets directly from the same `build_*_example()` functions used for training (stored under new names so Phase 4 is untouched).
2. **Base model was reloaded from scratch 3 times** inside `evaluate_adapter` (slow, and unnecessary GPU-OOM risk on smaller GPUs). Now the frozen base is loaded **once**, and all three LoRA adapters are hot-swapped in and out of it with `load_adapter()` / `set_adapter()` — the standard PEFT pattern for serving multiple adapters off one base model.
3. **No real "Base" row in the results table** — it was commented out because a `baseline_results` dict never actually existed. The fix recomputes the baseline **on the exact same model instance, tokenizer and prompts** used for the adapters (via `model.disable_adapter()`), so the comparison is genuinely apples-to-apples, and adds the row back automatically.
4. **EOS handling.** Qwen's chat template closes each turn with `<|im_end|>`, which isn't always the same token id as `tokenizer.eos_token_id`. If `.generate()` doesn't know to stop there, the model keeps rambling and that trailing text silently breaks exact-match SQL scoring. Generation now stops on both ids.
5. **Math generation length.** GSM8K answers are chain-of-thought and can run long; 256 new tokens was sometimes cutting off the final `#### <answer>` line before it appeared. Bumped to 320 tokens for math only.
6. **General robustness**: required variables are sanity-checked up front with clear error messages, and each evaluation run is wrapped in `try/finally` so GPU memory is always freed even if one task errors out mid-way.


In [48]:
import gc
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# --- fail fast with a clear message instead of a cryptic NameError later ---
for _name in ["tokenizer", "sql_test", "sam_test", "math_test",
              "SQL_ADAPTER_PATH", "SUMMARY_ADAPTER_PATH", "MATH_ADAPTER_PATH",
              "build_sql_example", "build_summary_example", "build_math_example"]:
    assert _name in globals(), (
        f"Missing '{_name}'. Run the data-loading and training cells above "
        f"before this evaluation section."
    )

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# FIX: rebuild eval prompts from the SAME builder functions used for
# training, instead of the earlier (differently-worded) prompt helpers.
# Stored under new names so Phase 4's router code (which reuses
# `sql_test_prompts` / `sum_test_prompts` / `math_test_prompts`) is
# unaffected.
sql_eval_prompts     = [build_sql_example(row)[0] for row in sql_test]
summary_eval_prompts = [build_summary_example(row)[0] for row in sam_test]
math_eval_prompts    = [build_math_example(row)[0] for row in math_test]

tokenizer.padding_side = "left"  # required for correct batched generation

# FIX: load the frozen base model ONCE and hot-swap adapters in and out of
# it (instead of calling from_pretrained 3 separate times).
print("Loading base model once for all adapter evaluations...")
_eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
_eval_base.eval()

eval_model = PeftModel.from_pretrained(_eval_base, SQL_ADAPTER_PATH, adapter_name="sql")
eval_model.load_adapter(SUMMARY_ADAPTER_PATH, adapter_name="summary")
eval_model.load_adapter(MATH_ADAPTER_PATH, adapter_name="math")
eval_model.eval()

# FIX: stop generation on BOTH tokenizer.eos_token_id and <|im_end|>
# (Qwen's chat-template turn-end token), since they aren't guaranteed to
# be the same id. Left unhandled, the model can keep generating past the
# real answer, which silently breaks exact-match SQL scoring.
_eos_ids = {tokenizer.eos_token_id}
_im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if _im_end_id is not None and _im_end_id != tokenizer.unk_token_id:
    _eos_ids.add(_im_end_id)
EOS_IDS = list(_eos_ids)

print("Stop token ids used for generation:", EOS_IDS)


Loading base model once for all adapter evaluations...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Stop token ids used for generation: [151645]


In [49]:
def generate_with_model(model, prompts, max_new_tokens=128, batch_size=8):
    all_outputs = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]

        conversations = [
            [{"role": "user", "content": prompt}]
            for prompt in batch_prompts
        ]

        texts = [
            tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=True
            )
            for conversation in conversations
        ]

        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True, max_length=1024
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=EOS_IDS,          # FIX: list of stop ids, not just one
            )

        input_length = inputs["input_ids"].shape[1]
        generated_tokens = outputs[:, input_length:]

        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        all_outputs.extend(decoded)

    return all_outputs


In [50]:
import re
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def normalize_sql(text):
    text = text.strip()
    text = re.sub(r"```(?:sql)?", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

def extract_gsm8k_answer(text):
    text = str(text).strip()
    if "####" in text:
        text = text.split("####")[-1]
    text = text.replace(",", "")
    match = re.findall(r"[-+]?\d+(?:\.\d+)?(?:/\d+)?", text)
    return match[-1] if match else None

def evaluate_sql(predictions, dataset):
    correct = sum(
        normalize_sql(p) == normalize_sql(row["answer"])
        for row, p in zip(dataset, predictions)
    )
    return correct / len(dataset)

def evaluate_summary(predictions, dataset):
    rouge1_scores, rougeL_scores = [], []
    for row, prediction in zip(dataset, predictions):
        scores = scorer.score(row["summary"], prediction or "")
        rouge1_scores.append(scores["rouge1"].fmeasure)
        rougeL_scores.append(scores["rougeL"].fmeasure)
    return {
        "rouge1": sum(rouge1_scores) / len(rouge1_scores),
        "rougeL": sum(rougeL_scores) / len(rougeL_scores),
    }

def evaluate_math(predictions, dataset):
    correct = 0
    for row, prediction in zip(dataset, predictions):
        expected = extract_gsm8k_answer(row["answer"])
        predicted = extract_gsm8k_answer(prediction)
        if expected is not None and predicted == expected:
            correct += 1
    return correct / len(dataset)


In [51]:
import contextlib

def run_full_eval(adapter_name):
    """
    adapter_name: "sql", "summary", "math", or None.
    None evaluates the frozen base model (adapters disabled) on THIS SAME
    model instance/tokenizer/prompts, so it is a true apples-to-apples
    baseline for the comparison table below.
    """
    label = adapter_name or "Base (no adapter)"
    print(f"\n{'='*50}\nTesting: {label}\n{'='*50}")

    try:
        if adapter_name is None:
            adapter_ctx = eval_model.disable_adapter()
        else:
            eval_model.set_adapter(adapter_name)
            adapter_ctx = contextlib.nullcontext()

        with adapter_ctx:
            sql_predictions = generate_with_model(
                eval_model, sql_eval_prompts, max_new_tokens=128, batch_size=8
            )
            sql_score = evaluate_sql(sql_predictions, sql_test)
            print(f"SQL Exact Match: {sql_score:.3f}")

            summary_predictions = generate_with_model(
                eval_model, summary_eval_prompts, max_new_tokens=128, batch_size=8
            )
            summary_scores = evaluate_summary(summary_predictions, sam_test)
            print(f"ROUGE-1: {summary_scores['rouge1']:.3f}  ROUGE-L: {summary_scores['rougeL']:.3f}")

            # FIX: more headroom for chain-of-thought math answers
            math_predictions = generate_with_model(
                eval_model, math_eval_prompts, max_new_tokens=320, batch_size=8
            )
            math_score = evaluate_math(math_predictions, math_test)
            print(f"Math Accuracy: {math_score:.3f}")

        return {
            "sql": sql_score,
            "summary_rouge1": summary_scores["rouge1"],
            "summary_rougeL": summary_scores["rougeL"],
            "math": math_score,
        }
    finally:
        # Always run, even if an assertion/OOM happens mid-evaluation.
        gc.collect()
        torch.cuda.empty_cache()


In [52]:
baseline_results        = run_full_eval(None)
sql_adapter_results     = run_full_eval("sql")
summary_adapter_results = run_full_eval("summary")
math_adapter_results    = run_full_eval("math")



Testing: Base (no adapter)
SQL Exact Match: 0.000
ROUGE-1: 0.344  ROUGE-L: 0.263
Math Accuracy: 0.330

Testing: sql
SQL Exact Match: 0.550
ROUGE-1: 0.345  ROUGE-L: 0.262
Math Accuracy: 0.300

Testing: summary
SQL Exact Match: 0.080
ROUGE-1: 0.445  ROUGE-L: 0.351
Math Accuracy: 0.300

Testing: math
SQL Exact Match: 0.050
ROUGE-1: 0.370  ROUGE-L: 0.291
Math Accuracy: 0.280


In [53]:
import pandas as pd

results_table = pd.DataFrame([
    {"Model": "Base",          **baseline_results},
    {"Model": "SQL-LoRA",      **sql_adapter_results},
    {"Model": "Summary-LoRA",  **summary_adapter_results},
    {"Model": "Math-LoRA",     **math_adapter_results},
])[["Model", "sql", "summary_rougeL", "math"]]

results_table.columns = ["Model", "SQL", "Summary ROUGE-L", "Math"]
results_table


,Model,SQL,Summary ROUGE-L,Math
0,Base,0.00,0.263283,0.33
1,SQL-LoRA,0.55,0.262268,0.30
2,Summary-LoRA,0.08,0.351380,0.30
3,Math-LoRA,0.05,0.291175,0.28


In [54]:
# Free GPU memory before Phase 4 loads its own router base model.
del eval_model
del _eval_base
gc.collect()
torch.cuda.empty_cache()
print("Eval model removed from GPU.")


Eval model removed from GPU.


Phase 4 — Train the Router

Input prompt
     ↓
Frozen Qwen2.5-0.5B
     ↓
Prompt embedding
     ↓
Linear Router
     ↓
3 probabilities
 ┌──────┬─────────┬──────┐
SQL   Summary    Math

In [55]:
!pip install -q scikit-learn

In [56]:
# del router_base
# gc.collect()
# torch.cuda.empty_cache()

In [57]:
import torch
import gc

from transformers import AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# router_base = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# router_base.eval()
# router_base.config.use_cache = False

# for param in router_base.parameters():
#     param.requires_grad = False

# print("Base model loaded and frozen.")
# print("Hidden size:", router_base.config.hidden_size)
router_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto"
)

router_base.eval()

for param in router_base.parameters():
    param.requires_grad = False

print("Router base loaded in FP32")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Router base loaded in FP32


Create the router dataset

We use the training prompts only to train the router.

In [58]:
router_train_texts = []
router_train_labels = []

# 0 = SQL
for prompt in [
    sql_prompt(row)
    for row in sql_train
]:
    router_train_texts.append(prompt)
    router_train_labels.append(0)

# 1 = Summary
for prompt in [
    summary_prompt(row)
    for row in sam_train
]:
    router_train_texts.append(prompt)
    router_train_labels.append(1)

# 2 = Math
for prompt in [
    math_prompt(row)
    for row in math_train
]:
    router_train_texts.append(prompt)
    router_train_labels.append(2)

print("Router training examples:", len(router_train_texts))
print("SQL:", router_train_labels.count(0))
print("Summary:", router_train_labels.count(1))
print("Math:", router_train_labels.count(2))

Router training examples: 3000
SQL: 1000
Summary: 1000
Math: 1000


Function to convert prompts → base-model embeddings

We use the frozen base model to create a representation of each prompt.

In [59]:
# def get_prompt_embeddings(
#     model,
#     prompts,
#     batch_size=16,
#     max_length=512
# ):
#     embeddings = []

#     for start in range(0, len(prompts), batch_size):

#         batch_prompts = prompts[
#             start:start + batch_size
#         ]

#         conversations = [
#             [{"role": "user", "content": prompt}]
#             for prompt in batch_prompts
#         ]

#         texts = [
#             tokenizer.apply_chat_template(
#                 conversation,
#                 tokenize=False,
#                 add_generation_prompt=True
#             )
#             for conversation in conversations
#         ]

#         inputs = tokenizer(
#             texts,
#             return_tensors="pt",
#             padding=True,
#             truncation=True,
#             max_length=max_length
#         )

#         inputs = {
#             k: v.to(model.device)
#             for k, v in inputs.items()
#         }

#         with torch.no_grad():
#             outputs = model(
#                 **inputs,
#                 output_hidden_states=True,
#                 use_cache=False
#             )

#         # Last transformer layer
#         hidden = outputs.hidden_states[-1]

#         # Mean-pool only real tokens
#         mask = inputs["attention_mask"].unsqueeze(-1)

#         pooled = (
#             hidden * mask
#         ).sum(dim=1) / mask.sum(dim=1)

#         embeddings.append(
#             pooled.float().cpu()
#         )

#     return torch.cat(embeddings, dim=0)

def get_prompt_embeddings(
    model,
    prompts,
    batch_size=8,
    max_length=512
):
    embeddings = []

    for start in range(0, len(prompts), batch_size):

        batch_prompts = prompts[start:start + batch_size]

        conversations = [
            [{"role": "user", "content": prompt}]
            for prompt in batch_prompts
        ]

        texts = [
            tokenizer.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=True
            )
            for conversation in conversations
        ]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            outputs = model(
                **inputs,
                output_hidden_states=True,
                use_cache=False
            )

        # Get final transformer hidden states
        hidden = outputs.hidden_states[-1].float()

        mask = inputs["attention_mask"].unsqueeze(-1).float()

        # Mean pooling
        pooled = (
            hidden * mask
        ).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        # Normalize embeddings
        pooled = torch.nn.functional.normalize(
            pooled,
            p=2,
            dim=1
        )

        embeddings.append(
            pooled.cpu()
        )

    return torch.cat(embeddings, dim=0)

In [60]:
# print(
#     "Train NaN:",
#     torch.isnan(router_train_embeddings).sum().item()
# )

# print(
#     "Val NaN:",
#     torch.isnan(router_val_embeddings).sum().item()
# )

# print(
#     "Test NaN:",
#     torch.isnan(router_test_embeddings).sum().item()
# )

In [61]:
#val embeding
router_val_texts = []
router_val_labels = []

for prompt in [
    sql_prompt(row)
    for row in sql_val
]:
    router_val_texts.append(prompt)
    router_val_labels.append(0)

for prompt in [
    summary_prompt(row)
    for row in sam_val
]:
    router_val_texts.append(prompt)
    router_val_labels.append(1)

for prompt in [
    math_prompt(row)
    for row in math_val
]:
    router_val_texts.append(prompt)
    router_val_labels.append(2)

router_val_embeddings = get_prompt_embeddings(
    router_base,
    router_val_texts,
    batch_size=16
)

router_val_labels = torch.tensor(
    router_val_labels,
    dtype=torch.long
)

router_train_labels = torch.tensor(
    router_train_labels,
    dtype=torch.long
)

print(router_val_embeddings.shape)


torch.Size([300, 896])


Build the actual router

Very small neural network:

In [69]:

# Force garbage collection and empty CUDA cache
gc.collect()
torch.cuda.empty_cache()

In [72]:
import gc, torch

for name in ["eval_model", "_eval_base", "sql_trainer", "sql_model",
             "summary_trainer", "summary_model", "math_trainer", "math_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1e9, "GB still allocated")

7.873808896 GB still allocated


In [73]:
import gc, torch

seen = set()
total = 0
tensors = []

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            t = obj if torch.is_tensor(obj) else obj.data
            if t.is_cuda and id(t) not in seen:
                seen.add(id(t))
                size_mb = t.element_size() * t.nelement() / 1e6
                total += size_mb
                tensors.append((size_mb, tuple(t.shape), t.dtype))
    except Exception:
        pass

tensors.sort(reverse=True)
print(f"Total CUDA tensors found: {len(tensors)}, total ≈ {total/1000:.2f} GB\n")
for size_mb, shape, dtype in tensors[:20]:
    print(f"{size_mb:8.1f} MB  shape={shape}  dtype={dtype}")

/usr/local/lib/python3.13/dist-packages/torch/__init__.py:1164: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/tmp/ipykernel_3678/1062643861.py:9: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):


Total CUDA tensors found: 353, total ≈ 7.88 GB

  4482.7 MB  shape=(16, 461, 151936)  dtype=torch.float32
   544.5 MB  shape=(151936, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  shape=(16, 512, 896)  dtype=torch.float32
    29.4 MB  sh

In [74]:
import sys, gc, torch

sys.last_traceback = None
sys.last_value = None
sys.last_type = None

gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1e9, "GB still allocated")

7.873808896 GB still allocated


In [75]:
def get_prompt_embeddings(model, prompts, batch_size=8, max_length=512):
    embeddings = []
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        conversations = [[{"role": "user", "content": p}] for p in batch_prompts]
        texts = [
            tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
            for c in conversations
        ]
        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            # FIX: go through the base transformer only (.model), not the
            # full CausalLM head — this skips computing per-token logits
            # over the whole 151936-token vocabulary, which was the
            # actual source of the OOM (a 4.5GB tensor per batch, unused).
            outputs = model.model(**inputs, use_cache=False)
            hidden = outputs.last_hidden_state.float()

        mask = inputs["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        embeddings.append(pooled.cpu())

        del outputs, hidden  # release explicitly before next batch

    return torch.cat(embeddings, dim=0)

In [76]:
import torch.nn as nn

class AdapterRouter(nn.Module):

    def __init__(self, hidden_size, num_adapters=3):
        super().__init__()

        self.classifier = nn.Linear(
            hidden_size,
            num_adapters
        )

    def forward(self, x):
        return self.classifier(x)


router = AdapterRouter(
    hidden_size=router_base.config.hidden_size,
    num_adapters=3
).to("cuda")

print(router)

AdapterRouter(
  (classifier): Linear(in_features=896, out_features=3, bias=True)
)


In [78]:
router_test_texts = []
router_test_labels = []

# SQL = 0
for prompt in sql_test_prompts:
    router_test_texts.append(prompt)
    router_test_labels.append(0)

# Summary = 1
for prompt in sum_test_prompts:
    router_test_texts.append(prompt)
    router_test_labels.append(1)

# Math = 2
for prompt in math_test_prompts:
    router_test_texts.append(prompt)
    router_test_labels.append(2)

router_test_embeddings = get_prompt_embeddings(
    router_base,
    router_test_texts,
    batch_size=16
)

router_test_labels = torch.tensor(
    router_test_labels,
    dtype=torch.long
)

In [79]:
# router_train_embeddings = get_prompt_embeddings(
#     router_base,
#     router_train_texts,
#     batch_size=16
# )

# print("Embedding shape:", router_train_embeddings.shape)
router_train_embeddings = get_prompt_embeddings(
    router_base,
    router_train_texts
)

router_val_embeddings = get_prompt_embeddings(
    router_base,
    router_val_texts
)

router_test_embeddings = get_prompt_embeddings(
    router_base,
    router_test_texts
)

In [80]:
from torch.utils.data import TensorDataset, DataLoader

router_train_dataset = TensorDataset(
    router_train_embeddings,
    router_train_labels
)

router_train_loader = DataLoader(
    router_train_dataset,
    batch_size=32,
    shuffle=True
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    router.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [81]:
EPOCHS = 10

for epoch in range(EPOCHS):

    router.train()

    total_loss = 0
    correct = 0
    total = 0

    for embeddings, labels in router_train_loader:

        embeddings = embeddings.to("cuda")
        labels = labels.to("cuda")

        optimizer.zero_grad()

        logits = router(embeddings)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_accuracy = correct / total

    # Validation
    router.eval()

    with torch.no_grad():

        val_logits = router(
            router_val_embeddings.to("cuda")
        )

        val_predictions = val_logits.argmax(dim=1)

        val_accuracy = (
            val_predictions.cpu()
            == router_val_labels
        ).float().mean().item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss {total_loss/len(router_train_loader):.4f} | "
        f"Train Acc {train_accuracy:.4f} | "
        f"Val Acc {val_accuracy:.4f}"
    )

Epoch 1/10 | Loss 0.9102 | Train Acc 0.9363 | Val Acc 1.0000
Epoch 2/10 | Loss 0.6199 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 3/10 | Loss 0.4366 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 4/10 | Loss 0.3197 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 5/10 | Loss 0.2430 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 6/10 | Loss 0.1906 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 7/10 | Loss 0.1536 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 8/10 | Loss 0.1263 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 9/10 | Loss 0.1060 | Train Acc 1.0000 | Val Acc 1.0000
Epoch 10/10 | Loss 0.0901 | Train Acc 1.0000 | Val Acc 1.0000


In [82]:
router.eval()

with torch.no_grad():

    test_logits = router(
        router_test_embeddings.to("cuda")
    )

    test_probabilities = torch.softmax(
        test_logits,
        dim=1
    )

    test_predictions = test_probabilities.argmax(
        dim=1
    ).cpu()

In [83]:
router_accuracy = (
    test_predictions == router_test_labels
).float().mean().item()

print(
    f"Router test accuracy: {router_accuracy:.4f}"
)

Router test accuracy: 1.0000


In [84]:
adapter_names = [
    "SQL-LoRA",
    "Summary-LoRA",
    "Math-LoRA"
]

for i in range(10):

    predicted = test_predictions[i].item()
    probabilities = test_probabilities[i].cpu().tolist()

    print("\nPrompt:")
    print(router_test_texts[i][:200])

    print(
        "SQL:     ",
        f"{probabilities[0]:.3f}"
    )

    print(
        "Summary: ",
        f"{probabilities[1]:.3f}"
    )

    print(
        "Math:    ",
        f"{probabilities[2]:.3f}"
    )

    print(
        "Selected:",
        adapter_names[predicted]
    )


Prompt:
Generate the SQL query that answers the question.

Database schema:
CREATE TABLE table_name_12 (vs_terran VARCHAR, vs_zerg VARCHAR)

Question:
What is the score vs. Terran when the score vs. Zerg is 6
SQL:      0.948
Summary:  0.029
Math:     0.023
Selected: SQL-LoRA

Prompt:
Generate the SQL query that answers the question.

Database schema:
CREATE TABLE table_name_99 (venue VARCHAR, away_team VARCHAR)

Question:
Where was the game played where Collingwood was the away te
SQL:      0.960
Summary:  0.020
Math:     0.020
Selected: SQL-LoRA

Prompt:
Generate the SQL query that answers the question.

Database schema:
CREATE TABLE table_22180353_1 (construction VARCHAR, registration VARCHAR)

Question:
What is every construction date for the regist
SQL:      0.965
Summary:  0.020
Math:     0.015
Selected: SQL-LoRA

Prompt:
Generate the SQL query that answers the question.

Database schema:
CREATE TABLE table_name_68 (effic INTEGER, avg_g VARCHAR)

Question:
What is the highest eff

In [85]:
# ── Phase 5 — End-to-end routed inference ─────────────────────────────
# Prompt → router → selected LoRA adapter → Qwen → response

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM
from peft import PeftModel

for _name in ["router", "router_base", "tokenizer",
              "SQL_ADAPTER_PATH", "SUMMARY_ADAPTER_PATH", "MATH_ADAPTER_PATH",
              "get_prompt_embeddings"]:
    assert _name in globals(), f"Missing '{_name}'. Run Phase 2–4 cells first."

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_LABELS = ["sql", "summary", "math"]  # index order must match router training labels (0=SQL,1=Summary,2=Math)

tokenizer.padding_side = "left"

# Reuse eval_model from Phase 3 if it's still around; otherwise load fresh.
if "eval_model" not in globals():
    print("Loading base model + adapters for routed inference...")
    _infer_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto"
    )
    _infer_base.eval()
    eval_model = PeftModel.from_pretrained(_infer_base, SQL_ADAPTER_PATH, adapter_name="sql")
    eval_model.load_adapter(SUMMARY_ADAPTER_PATH, adapter_name="summary")
    eval_model.load_adapter(MATH_ADAPTER_PATH, adapter_name="math")
    eval_model.eval()

_eos_ids = {tokenizer.eos_token_id}
_im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if _im_end_id is not None and _im_end_id != tokenizer.unk_token_id:
    _eos_ids.add(_im_end_id)
EOS_IDS = list(_eos_ids)


def classify_prompt(prompt):
    """Use the trained router to pick which adapter fits this prompt."""
    embedding = get_prompt_embeddings(router_base, [prompt], batch_size=1)
    router.eval()
    with torch.no_grad():
        logits = router(embedding.to(next(router.parameters()).device))
        probs = F.softmax(logits, dim=1)[0].cpu().tolist()
    predicted_idx = int(max(range(3), key=lambda i: probs[i]))
    return ADAPTER_LABELS[predicted_idx], dict(zip(ADAPTER_LABELS, probs))


def generate_response(prompt, adapter_name, max_new_tokens=256):
    eval_model.set_adapter(adapter_name)
    conversation = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        output = eval_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=EOS_IDS,
        )
    generated = output[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()


def route_and_generate(prompt, verbose=True):
    adapter_name, probs = classify_prompt(prompt)
    response = generate_response(prompt, adapter_name)
    if verbose:
        print(f"Prompt: {prompt[:120]}{'...' if len(prompt) > 120 else ''}")
        print("Router probs: " + ", ".join(f"{k}={v:.3f}" for k, v in probs.items()))
        print(f"Selected adapter: {adapter_name}")
        print(f"Response:\n{response}\n")
    return {"adapter": adapter_name, "probabilities": probs, "response": response}

Loading base model + adapters for routed inference...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [87]:
test_examples = {
    "SQL":     sql_prompt(sql_test[10]),
    "Summary": summary_prompt(sam_test[10]),
    "Math":    math_prompt(math_test[10]),
}

for task, prompt in test_examples.items():
    print(f"{'='*60}\nTask: {task}\n{'='*60}")
    route_and_generate(prompt)

Task: SQL
Prompt: Generate the SQL query that answers the question.

Database schema:
CREATE TABLE table_name_87 (years VARCHAR, league VA...
Router probs: sql=0.961, summary=0.018, math=0.021
Selected adapter: sql
Response:
SELECT years FROM table_name_87 WHERE league = "400 (21)"

Task: Summary
Prompt: Summarize the following conversation in 1-3 concise sentences.

Conversation:
Wanda: Let's make a party!
Gina: Why?
Wand...
Router probs: sql=0.017, summary=0.939, math=0.044
Selected adapter: summary
Response:
Wanda wants to have a party on Friday. She needs Gina's help to prepare a list of things for it.

Task: Math
Prompt: Solve the following math word problem.

Problem:
A new program had 60 downloads in the first month. The number of downlo...
Router probs: sql=0.076, summary=0.089, math=0.835
Selected adapter: math
Response:
The number of downloads in the second month is 60*3=<<60*3=180>>180.
In the third month, it decreased to 180*(1-30/100)=<<180*(1-30)/100=54>>54.
So, the total

In [88]:
torch.save(
    router.state_dict(),
    "/content/adapter_router.pt"
)

In [89]:
results_table.to_csv(
    "/content/specialization_results.csv",
    index=False
)

In [90]:
!pip install -q huggingface_hub

In [91]:
from huggingface_hub import login

login()

In [94]:
from huggingface_hub import HfApi

HF_USERNAME = "nithinai517"

SQL_ADAPTER_PATH = "/content/adapters/sql"
SUMMARY_ADAPTER_PATH = "/content/adapters/summary"
MATH_ADAPTER_PATH = "/content/adapters/math"

SQL_REPO = f"{HF_USERNAME}/qwen-sql-lora"
SUMMARY_REPO = f"{HF_USERNAME}/qwen-summary-lora"
MATH_REPO = f"{HF_USERNAME}/qwen-math-lora"

api = HfApi()

print("SQL:    ", SQL_REPO)
print("Summary:", SUMMARY_REPO)
print("Math:   ", MATH_REPO)

SQL:     nithinai517/qwen-sql-lora
Summary: nithinai517/qwen-summary-lora
Math:    nithinai517/qwen-math-lora


In [95]:
import os

for name, path in [
    ("SQL", SQL_ADAPTER_PATH),
    ("Summary", SUMMARY_ADAPTER_PATH),
    ("Math", MATH_ADAPTER_PATH),
]:
    print(f"\n{name} adapter:")

    if os.path.exists(path):
        print("Path exists:", path)
        print("Files:")
        for file in os.listdir(path):
            print("  ", file)
    else:
        print("ERROR: Path does not exist:", path)


SQL adapter:
Path exists: /content/adapters/sql
Files:
   tokenizer.json
   README.md
   tokenizer_config.json
   adapter_model.safetensors
   adapter_config.json
   chat_template.jinja

Summary adapter:
Path exists: /content/adapters/summary
Files:
   tokenizer.json
   README.md
   tokenizer_config.json
   adapter_model.safetensors
   adapter_config.json
   chat_template.jinja

Math adapter:
Path exists: /content/adapters/math
Files:
   tokenizer.json
   README.md
   tokenizer_config.json
   adapter_model.safetensors
   adapter_config.json
   chat_template.jinja


In [96]:
# from huggingface_hub import HfApi

# api = HfApi()

# api.create_repo(
#     repo_id="nithinai517/qwen-sql-lora",
#     repo_type="model",
#     exist_ok=True
# )

# api.upload_folder(
#     folder_path="./qwen-sql-lora",
#     repo_id="nithinai517/qwen-sql-lora",
#     repo_type="model"
# )

In [97]:
api.create_repo(
    repo_id=SQL_REPO,
    repo_type="model",
    exist_ok=True
)

api.create_repo(
    repo_id=SUMMARY_REPO,
    repo_type="model",
    exist_ok=True
)

api.create_repo(
    repo_id=MATH_REPO,
    repo_type="model",
    exist_ok=True
)

print("All 3 Hugging Face repositories created.")

All 3 Hugging Face repositories created.


In [98]:
api.upload_folder(
    folder_path=SQL_ADAPTER_PATH,
    repo_id=SQL_REPO,
    repo_type="model"
)

print("SQL adapter uploaded successfully.")

SQL adapter uploaded successfully.


In [99]:
api.upload_folder(
    folder_path=SUMMARY_ADAPTER_PATH,
    repo_id=SUMMARY_REPO,
    repo_type="model"
)

print("Summary adapter uploaded successfully.")

Summary adapter uploaded successfully.


In [100]:
api.upload_folder(
    folder_path=MATH_ADAPTER_PATH,
    repo_id=MATH_REPO,
    repo_type="model"
)

print("Math adapter uploaded successfully.")

Math adapter uploaded successfully.


In [101]:
print("SQL adapter files:")
for file in api.list_repo_files(
    repo_id=SQL_REPO,
    repo_type="model"
):
    print(file)

SQL adapter files:
.gitattributes
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json


In [102]:
router

AdapterRouter(
  (classifier): Linear(in_features=896, out_features=3, bias=True)
)

In [103]:
import torch

ROUTER_PATH = "/content/adapter_router.pt"

torch.save(
    router.state_dict(),
    ROUTER_PATH
)

print("Router saved:", ROUTER_PATH)

Router saved: /content/adapter_router.pt


In [104]:
import os

print("Exists:", os.path.exists(ROUTER_PATH))

if os.path.exists(ROUTER_PATH):
    size_mb = os.path.getsize(ROUTER_PATH) / (1024 * 1024)
    print(f"Router size: {size_mb:.2f} MB")

Exists: True
Router size: 0.01 MB


In [105]:
from google.colab import files

files.download(ROUTER_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [106]:
ADAPTER_REPOS = {
    "sql": SQL_REPO,
    "summary": SUMMARY_REPO,
    "math": MATH_REPO,
}

ADAPTER_REPOS

{'sql': 'nithinai517/qwen-sql-lora',
 'summary': 'nithinai517/qwen-summary-lora',
 'math': 'nithinai517/qwen-math-lora'}

In [107]:
from huggingface_hub import snapshot_download

test_sql_path = snapshot_download(
    repo_id=SQL_REPO,
    repo_type="model"
)

print("Downloaded SQL adapter to:")
print(test_sql_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Downloaded SQL adapter to:
/root/.cache/huggingface/hub/models--nithinai517--qwen-sql-lora/snapshots/b16efafc1769fa9ef4c73be1fb19278edd3b1963


In [108]:
import os

for file in os.listdir(test_sql_path):
    print(file)

tokenizer.json
.gitattributes
README.md
tokenizer_config.json
adapter_model.safetensors
adapter_config.json
chat_template.jinja


In [109]:
print("=" * 60)
print("FINAL FINE-TUNING ARTIFACT STATUS")
print("=" * 60)

print("\nSQL Adapter")
print(" Local :", SQL_ADAPTER_PATH)
print(" HF    :", SQL_REPO)

print("\nSummary Adapter")
print(" Local :", SUMMARY_ADAPTER_PATH)
print(" HF    :", SUMMARY_REPO)

print("\nMath Adapter")
print(" Local :", MATH_ADAPTER_PATH)
print(" HF    :", MATH_REPO)

print("\nRouter")
print(" Local :", ROUTER_PATH)

print("\nEverything ready for GitHub + Colab demo.")

FINAL FINE-TUNING ARTIFACT STATUS

SQL Adapter
 Local : /content/adapters/sql
 HF    : nithinai517/qwen-sql-lora

Summary Adapter
 Local : /content/adapters/summary
 HF    : nithinai517/qwen-summary-lora

Math Adapter
 Local : /content/adapters/math
 HF    : nithinai517/qwen-math-lora

Router
 Local : /content/adapter_router.pt

Everything ready for GitHub + Colab demo.
